# Tome 7, Chapter 5: The Fourier Transform (Python companion)

Exercises first, solutions at the end. The Fourier transform on a computer: sampling, aliasing and audio, Shannon reconstruction in practice, Hermite functions inside the DFT, and the Cauchy law that averaging cannot tame. Requirements: numpy and matplotlib.

*Before each solution, predict what the output will look like.*

## Exercises

**P5.1 (★★) CS application: sampling, aliasing and audio.** Synthesise one second of a chirp whose frequency rises from $100$ Hz to $8000$ Hz, sampled at $44\,100$ Hz. Downsample it to $8000$ Hz in two ways: by keeping one sample in $5.5125$ (interpolate linearly), and by first removing all frequencies above $4000$ Hz with the FFT. Compute the spectrogram of each result (short-time FFT) and explain what you see with the Nyquist rate of Theorem 5.10. Write the three signals as WAV files and listen.

**P5.2 (★) Shannon in practice.** Let $f\in PW_{1/2}$ be a random combination of shifted sinc functions. Reconstruct $f$ on $[-20,20]$ from its samples at the integers with the series of Theorem 5.10 truncated to $|n|\le N$, and plot the error against $N$. Then oversample: with samples at spacing $\frac12$ and a smoother interpolation kernel, how does the error behave?

**P5.3 (★) Hermite functions inside the DFT.** The $N\times N$ unitary DFT matrix $F_N$ satisfies $F_N^4=I$, so its eigenvalues lie in $\{1,-i,-1,i\}$. Compute their multiplicities for several $N$. Sample $h_0,\dots,h_5$ at the points $\frac{k-N/2}{\sqrt N}$ and check that the centred DFT maps each sample vector approximately to $(-i)^n$ times itself (Theorem 5.5).

**P5.4 (★) When averaging fails.** Compare the running means of $10^5$ independent Gaussian and Cauchy variables. Estimate the characteristic function of the Cauchy law from samples and compare it with $e^{-|t|}$ (Proposition 5.11). Check Corollary 5.12 by plotting the histogram of means of $1000$ Cauchy variables against the Cauchy density.

## Solutions

### P5.1

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import wave

def write_wav(name, x, rate):
    y = np.int16(np.clip(x / np.max(np.abs(x)), -1, 1) * 32000)
    with wave.open(name, 'wb') as w:
        w.setnchannels(1); w.setsampwidth(2); w.setframerate(rate); w.writeframes(y.tobytes())

R = 44100
t = np.arange(R) / R
f0, f1 = 100.0, 8000.0
phase = 2 * np.pi * (f0 * t + (f1 - f0) * t ** 2 / 2)   # instantaneous frequency f0 + (f1 - f0) t
x = np.sin(phase)

r = 8000
tn = np.arange(r) / r
naive = np.interp(tn, t, x)                               # keep the values at the new instants
X = np.fft.rfft(x); freqs = np.fft.rfftfreq(R, 1 / R)
X[freqs > r / 2] = 0                                      # ideal low-pass filter at the new Nyquist frequency
filtered = np.interp(tn, t, np.fft.irfft(X, R))

def spectrogram(sig, rate, win=256, hop=64):
    w = np.hanning(win)
    frames = np.array([sig[i:i + win] * w for i in range(0, len(sig) - win, hop)])
    return np.abs(np.fft.rfft(frames, axis=1)).T, np.fft.rfftfreq(win, 1 / rate)

fig, ax = plt.subplots(1, 3, figsize=(12, 3.2))
for a, (sig, rate, title) in zip(ax, [(x, R, '44.1 kHz original'), (naive, r, '8 kHz, no filter'), (filtered, r, '8 kHz, filtered first')]):
    S, fr = spectrogram(sig, rate, win=1024 if rate == R else 256, hop=256 if rate == R else 64)
    a.imshow(20 * np.log10(S + 1e-6), origin='lower', aspect='auto', extent=[0, 1, 0, fr[-1]], vmin=-20, cmap='magma')
    a.set_title(title); a.set_xlabel('time (s)')
ax[0].set_ylabel('frequency (Hz)'); plt.show()
for name, sig, rate in [('chirp_44k.wav', x, R), ('chirp_8k_aliased.wav', naive, r), ('chirp_8k_filtered.wav', filtered, r)]:
    write_wav(name, sig, rate)
print("wrote three WAV files")

In the original, the frequency rises linearly to $8000$ Hz. Sampled at $8000$ Hz, only frequencies up to the Nyquist frequency $4000$ Hz can be represented (Theorem 5.10 with $W=4000$, rate $2W$). Without filtering, the second half of the chirp *folds back*: a tone at $4000+\delta$ Hz produces exactly the samples of a tone at $4000-\delta$ Hz, so the spectrogram shows a rising line that turns into a falling one, and you hear a whistle going up and then down. With the low-pass filter first, the second half simply goes silent: information is lost, but nothing false is added. This is why every analogue-to-digital converter has an anti-aliasing filter in front of it, and why CDs sample at $44.1$ kHz rather than $40$ kHz (Tome 4, Chapter 8). The ideal filter used here (zeroing FFT bins) is itself the projection onto a Paley–Wiener space, made periodic.

### P5.2

In [ ]:
rng = np.random.default_rng(3)
centres = rng.uniform(-10, 10, 12); amps = rng.standard_normal(12)
f = lambda x: sum(a * np.sinc(x - c) for a, c in zip(amps, centres))   # np.sinc(t) = sin(pi t)/(pi t); f is in PW_{1/2}
xs = np.linspace(-20, 20, 4001)
errs = []
for N in [10, 20, 40, 80, 160, 320]:
    n = np.arange(-N, N + 1)
    rec = np.sinc(xs[:, None] - n[None, :]) @ f(n)
    errs.append(np.max(np.abs(rec - f(xs))))
    print(f"N={N:4d}: max error on [-20,20] = {errs[-1]:.2e}")
# oversampling: samples at spacing 1/2; a kernel whose transform is 1 on [-1/2,1/2] and decays smoothly to 0 at |xi| = 1
def smooth_kernel(x):
    # transform = trapezoid: 1 on [-1/2,1/2], linear to 0 at |xi|=1; its inverse transform is
    # (2 * the convolution of the indicators of [-3/4,3/4] and [-1/4,1/4]), that is
    small = np.abs(x) < 1e-6
    xs_ = np.where(small, 1.0, x)
    return np.where(small, 1.5, (np.cos(np.pi * xs_) - np.cos(2 * np.pi * xs_)) / (np.pi ** 2 * xs_ ** 2))
for N in [10, 20, 40, 80]:
    n = np.arange(-2 * N, 2 * N + 1) / 2
    rec = 0.5 * smooth_kernel(xs[:, None] - n[None, :]) @ f(n)
    print(f"oversampled x2, |n/2| <= {N:3d}: max error = {np.max(np.abs(rec - f(xs))):.2e}")

With samples exactly at the Nyquist rate, the truncated sinc series converges, but slowly: the error decreases only like $\frac1N$ or so, because $\operatorname{sinc}$ decays like $\frac1{|x|}$ and the far samples still matter. Sampling twice as fast buys freedom: the interpolation kernel no longer needs a transform equal to the indicator of $[-\frac12,\frac12]$; any kernel whose transform is $1$ on $[-\frac12,\frac12]$ and vanishes outside $[-1,1]$ works (the copies created by sampling at rate $2$ sit at distance $2$ and do not overlap). Choosing a transform that falls off as a continuous trapezoid gives a kernel decaying like $\frac1{x^2}$, and the error drops much faster with $N$. Smoother transitions give faster decay still. Every practical resampler, in audio software or in image scaling, is a version of this trade between sampling rate and kernel length.

### P5.3

In [ ]:
from math import factorial
for N in [16, 17, 64, 101, 256]:
    F = np.fft.fft(np.eye(N)) / np.sqrt(N)
    ev = np.linalg.eigvals(F)
    counts = {name: int(np.sum(np.abs(ev - z) < 1e-6)) for name, z in [('1', 1), ('-i', -1j), ('-1', -1), ('i', 1j)]}
    print(f"N={N:4d}: multiplicities {counts}")
N = 256
k = np.arange(N); x = (k - N / 2) / np.sqrt(N)
def hermite(n, x):
    P = np.poly1d([1.0])
    for _ in range(n):
        P = np.poly1d([4 * np.pi, 0]) * P - P.deriv()
    h = P(x) * np.exp(-np.pi * x ** 2)
    return h / np.sqrt(np.sum(h ** 2))
for n in range(6):
    v = hermite(n, x)
    Fv = np.fft.fftshift(np.fft.fft(np.fft.ifftshift(v))) / np.sqrt(N)   # centred unitary DFT
    print(f"h_{n}: |F v - (-i)^{n} v| = {np.linalg.norm(Fv - (-1j) ** n * v):.2e}")

The DFT matrix has eigenvalues $1,-i,-1,i$ with multiplicities close to $\frac N4$ each, depending on $N$ modulo $4$ (a classical result going back to Gauss's evaluation of quadratic Gauss sums, Chapter 7). With the grid $\frac{k-N/2}{\sqrt N}$, the DFT approximates the continuous transform: the step in $x$ is $\frac1{\sqrt N}$, the range is $\sqrt N$, and these match the step and range in $\xi$. Sampled Hermite functions are then eigenvectors of the DFT to machine precision for small $n$ (the only errors come from the Gaussian tails beyond the grid, of size about $e^{-\pi N/4}$): the discrete world inherits Theorem 5.5. This is the starting point of the discrete fractional Fourier transform and of the "Hermite–Gaussian" beams of optics, which keep their shape under propagation.

### P5.4

In [ ]:
n = 100_000
G = rng.standard_normal(n); C = rng.standard_cauchy(n)
idx = np.arange(1, n + 1)
fig, ax = plt.subplots(1, 3, figsize=(13, 3.2))
ax[0].semilogx(idx, np.cumsum(G) / idx, label='Gaussian'); ax[0].semilogx(idx, np.cumsum(C) / idx, label='Cauchy')
ax[0].legend(); ax[0].set_title('running means')
ts = np.linspace(-5, 5, 201)
emp = np.array([np.mean(np.cos(t * C)) for t in ts])   # the imaginary part vanishes by symmetry
ax[1].plot(ts, emp, label='empirical'); ax[1].plot(ts, np.exp(-np.abs(ts)), '--', label='exp(-|t|)'); ax[1].legend()
means = rng.standard_cauchy((20_000, 1000)).mean(axis=1)
bins = np.linspace(-10, 10, 201)
ax[2].hist(means, bins=bins, density=True, alpha=0.6, label='means of 1000')
ax[2].plot(bins, 1 / (np.pi * (1 + bins ** 2)), 'k', label='Cauchy density'); ax[2].legend(); plt.show()
print("max deviation of empirical characteristic function:", np.max(np.abs(emp - np.exp(-np.abs(ts)))))
print("fraction of means of 1000 Cauchy variables with |mean| > 10:", np.mean(np.abs(means) > 10), " theory:", 1 - 2 / np.pi * np.arctan(10))

The Gaussian running mean settles at $0$ with fluctuations of order $\frac1{\sqrt n}$; the Cauchy running mean never settles: every now and then one huge observation (the law has tails $\frac1{\pi x^2}$, so values beyond $x$ have probability about $\frac2{\pi x}$) moves it by a jump of order one, forever. The empirical characteristic function matches $e^{-|t|}$ to within about $0.005$, with the corner at $0$ clearly visible: a characteristic function with a corner at the origin means an infinite mean, since a finite mean would make it differentiable there. And the means of $1000$ Cauchy variables have exactly the Cauchy law, as Corollary 5.12 says: about $6\%$ of them exceed $10$ in absolute value, the same fraction as for a single variable. For heavy-tailed data such as network traffic or file sizes, the median and other robust statistics replace the mean for exactly this reason.